In [1]:
import os

In [2]:
! pip install youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [3]:
# YouTube Transcript
from youtube_transcript_api import YouTubeTranscriptApi

# LangChain
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Environment Variables
from dotenv import load_dotenv

# Tokenization
import tiktoken

from youtube_transcript_api._errors import TranscriptsDisabled

C:\Users\shakt\AppData\Local\Temp\ipykernel_22412\1588051477.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Step1 a MARKDOWN

In [4]:
# video_id = "Gfr50f6ZBvo" # only the ID, not full URL
# try:
#     # If you don’t care which language, this returns the “best” one
#     transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])

#     # Flatten it to plain text
#     transcript = " ".join(chunk["text"] for chunk in transcript_list)
#     print(transcript)

# except TranscriptsDisabled:
#     print("No captions available for this video.")

In [5]:

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    # Create API instance
    api = YouTubeTranscriptApi()

    # Fetch transcript
    transcript_list = api.fetch(video_id, languages=['en'])

    # Flatten into plain text
    transcript = " ".join(
        chunk.text for chunk in transcript_list
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

except Exception as e:
    print(f"Error: {e}")


the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [6]:
transcript_list = api.fetch(
    video_id,
    languages=["en"]
)

transcript = " ".join(
    chunk.text for chunk in transcript_list
)

print("Total snippets:", len(transcript_list))
print("Total characters:", len(transcript))
print("Total words:", len(transcript.split()))

Total snippets: 3790
Total characters: 133836
Total words: 25382


STEP 2 - TEXT SPLITTER

In [7]:
splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000 ,
    chunk_overlap=200)
chunks= splitter.create_documents([transcript])

In [8]:
len(chunks)

168

In [9]:
chunks[0]

Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inter

In [10]:
print("Total chunks:", len(chunks))

print("First chunk length:",
      len(chunks[0].page_content))

print("Total document characters:",
      len(transcript))

Total chunks: 168
First chunk length: 999
Total document characters: 133836


Step 3- INDEXING (EMBEDDING GENERATION AND STORI NG IN VECTOR STORES)

In [13]:
embeddings= OpenAIEmbeddings(model='text-embedding-3-small')
vector_store= FAISS.from_documents(chunks,embeddings)
vector_store

In [14]:
vector_store.index_to_docstore_id

{0: 'a1f00310-47b3-4661-b3e3-419b6fb6a0e4',
 1: '54e60960-1d80-4b7e-85ac-a310808aadb4',
 2: '9eccbc2c-4a0a-4124-b839-a5f02c07c0bb',
 3: '092e9f99-9009-4f01-b5f8-c68827d6f63b',
 4: 'f262b2d4-24d7-4f8d-95ee-543680d27bcb',
 5: '4426bfb8-9f34-44ec-97bf-00ef031b3a6a',
 6: '2ab23646-fff6-40ea-ba96-f4c3a7762729',
 7: '82177847-2549-49d1-af08-11458f97e626',
 8: '9c6e10de-a6c7-4e7d-8830-f992f21136b5',
 9: '0a50cec8-cbfd-4ad2-b6f5-2e2772bb749b',
 10: 'f279c175-1632-4a34-8b0d-024ae0d62ba4',
 11: '7f650de7-86f5-466b-8159-eb8abadf0c03',
 12: '0b0c5cf7-ddad-41ee-824e-1d56addc9b5f',
 13: '740c9595-d645-406e-a6ea-fe40c2a87254',
 14: '3c97228c-cb10-43a0-a0cf-66b4480cd303',
 15: '8716d538-086a-4ee4-9eef-40ec3234e350',
 16: '7fee57e4-07d7-4c2d-97ed-04547af34d0f',
 17: '6283eb94-9333-4665-aecd-e6f8978ea18c',
 18: 'de81c0fc-feae-43b4-bcbc-54c229856796',
 19: 'c4b33749-946f-4a41-9029-4699c3da1c76',
 20: '6e98b3c3-89fe-4185-8052-1a48a346a62f',
 21: '32b99c99-808a-4848-a56e-b829fdf39407',
 22: 'f2791af0-d9c4-

In [15]:
retriever= vector_store.as_retriever(search_type='similarity' , search_kwargs={'k':4})

In [16]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B229CA2490>, search_kwargs={'k': 4})

In [17]:
retriever.invoke('what is deepmind')

[Document(id='a1f00310-47b3-4661-b3e3-419b6fb6a0e4', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu